<a href="https://colab.research.google.com/github/elsa-paul11/de-portfolio-2026/blob/main/module-03-airflow/notebooks/m3_d4_airflow_dags.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install apache-airflow==2.9.3 -q

In [2]:
import os
os.environ["AIRFLOW_HOME"] = "/content/airflow"
print("✅ Environment set")

✅ Environment set


In [3]:
import os
db_exists = os.path.exists("/content/airflow/airflow.db")
print(f"Airflow DB exists: {db_exists}")

Airflow DB exists: True


In [4]:
!airflow db migrate

DB: sqlite:////content/airflow/airflow.db
Performing upgrade to the metadata database sqlite:////content/airflow/airflow.db
[2026-07-07T16:22:46.655+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.schemas
[2026-07-07T16:22:46.656+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.tables
[2026-07-07T16:22:46.657+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.types
[2026-07-07T16:22:46.657+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.constraints
[2026-07-07T16:22:46.657+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.defaults
[2026-07-07T16:22:46.657+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.comments
[2026-07-07T16:22:46.868+0000] {migration.py:210} INFO - Context impl SQLiteImpl.
[2026-07-07T16:22:46.868+0000] {migration.py:213} INFO - Will assume non-transactional DDL.
[2026-07-07T16:22:46.870+0000] {db.py:1625} INFO - Creating tables
INFO  [alembic.runtime.migration] Context impl SQLiteImp

In [5]:
import os
os.environ["AIRFLOW_HOME"] = "/content/airflow"

In [6]:
dag_code = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import logging

logger = logging.getLogger("airflow.task")

default_args = {
    "owner": "elsa",
    "retries": 3,
    "retry_delay": timedelta(minutes=2),
}

with DAG(
    dag_id="orders_pipeline_v1",
    default_args=default_args,
    description="Extract, validate, transform orders data",
    schedule_interval="@daily",
    start_date=datetime(2026, 6, 1),
    catchup=False,
    tags=["module3", "day4"],
) as dag:

    def extract_orders(**context):
        logger.info("Extracting orders from source...")
        record_count = 10000
        logger.info(f"Extracted {record_count} records")
        return record_count

    def validate_data(**context):
        logger.info("Validating extracted data...")
        record_count = context["ti"].xcom_pull(task_ids="extract_orders")
        if record_count == 0:
            raise ValueError("Validation failed: zero records extracted")
        logger.info(f"Validation passed | records={record_count}")

    def transform_and_load(**context):
        logger.info("Transforming and writing to S3 (idempotent overwrite)...")
        logger.info("Write complete | mode=overwrite | partition=event_date")

    def send_notification(**context):
        logger.info("Pipeline complete. Notification sent.")

    t1 = PythonOperator(task_id="extract_orders", python_callable=extract_orders)
    t2 = PythonOperator(task_id="validate_data", python_callable=validate_data)
    t3 = PythonOperator(task_id="transform_and_load", python_callable=transform_and_load)
    t4 = PythonOperator(task_id="send_notification", python_callable=send_notification)

    t1 >> t2 >> t3 >> t4

'''

os.makedirs("/content/airflow/dags", exist_ok=True)
with open("/content/airflow/dags/orders_pipeline.py", "w") as f:
    f.write(dag_code)

print("✅ DAG file written")

✅ DAG file written


In [7]:
!airflow dags list

[2026-07-07T16:22:57.594+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.schemas
[2026-07-07T16:22:57.595+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.tables
[2026-07-07T16:22:57.595+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.types
[2026-07-07T16:22:57.595+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.constraints
[2026-07-07T16:22:57.595+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.defaults
[2026-07-07T16:22:57.595+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.comments
Error: Failed to load all files. For details, run `airflow dags 
list-import-errors`
dag_id                       | fileloc                     | owners  | is_paused
=============================+=============================+=========+==========
conditional_dataset_and_time | /usr/local/lib/python3.12/d | airflow | True     
_based_timetable             | ist-packages/airflow/exampl |         |          
            

In [8]:
!airflow dags list-import-errors

[2026-07-07T16:23:04.050+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.schemas
[2026-07-07T16:23:04.052+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.tables
[2026-07-07T16:23:04.052+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.types
[2026-07-07T16:23:04.052+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.constraints
[2026-07-07T16:23:04.052+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.defaults
[2026-07-07T16:23:04.053+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.comments
filepath                               | error                                  
=======================================+========================================
/usr/local/lib/python3.12/dist-package | Traceback (most recent call last):     
s/airflow/example_dags/tutorial_object |   File                                 
storage.py                             | "/usr/local/lib/python3.12/dist-package
                

In [9]:
!airflow dags test orders_pipeline_v1 2026-07-07

[2026-07-07T16:23:09.932+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.schemas
[2026-07-07T16:23:09.933+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.tables
[2026-07-07T16:23:09.933+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.types
[2026-07-07T16:23:09.934+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.constraints
[2026-07-07T16:23:09.934+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.defaults
[2026-07-07T16:23:09.934+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.comments
[2026-07-07T16:23:10.124+0000] {dagbag.py:545} INFO - Filling up the DagBag from /content/airflow/dags
[2026-07-07T16:23:10.472+0000] {example_kubernetes_executor.py:39} WARNING - The example_kubernetes_executor example DAG requires the kubernetes provider. Please install it with: pip install apache-airflow[cncf.kubernetes]
[2026-07-07T16:23:10.479+0000] {example_python_decorator.py:80} WARNING - The virtalenv_python 

In [13]:
import os
os.environ["AIRFLOW_HOME"] = "/content/airflow"

# Reset counter file before writing new DAG
counter_file = "/tmp/flaky_extract_attempts.txt"
if os.path.exists(counter_file):
    os.remove(counter_file)
    print("Counter reset")

dag_code_v2 = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import logging
import os

logger = logging.getLogger("airflow.task")

default_args = {
    "owner": "kavya",
    "retries": 2,
    "retry_delay": timedelta(seconds=5),
}

with DAG(
    dag_id="orders_pipeline_v2",
    default_args=default_args,
    description="Pipeline with failure simulation",
    schedule_interval="@daily",
    start_date=datetime(2026, 6, 1),
    catchup=False,
    tags=["module3", "day4"],
) as dag:

    def flaky_extract(**context):
        counter_file = "/tmp/flaky_extract_attempts.txt"

        # Read current attempt count
        if os.path.exists(counter_file):
            with open(counter_file, "r") as f:
                attempt = int(f.read().strip()) + 1
        else:
            attempt = 1

        # Write updated count
        with open(counter_file, "w") as f:
            f.write(str(attempt))

        logger.info(f"Extract attempt #{attempt}")

        if attempt < 3:
            logger.error(f"Connection timeout on attempt #{attempt}")
            raise ConnectionError(f"MySQL connection failed on attempt #{attempt}")

        # Success on attempt 3
        os.remove(counter_file)
        logger.info(f"Connection succeeded on attempt #{attempt}")
        return 10000

    def validate_data(**context):
        record_count = context["ti"].xcom_pull(task_ids="flaky_extract")
        logger.info(f"Validation passed | records={record_count}")

    def transform_and_load(**context):
        execution_date = context["execution_date"]
        logger.info(f"Writing partition for: {execution_date.date()}")
        logger.info("Write complete | mode=overwrite | idempotent=True")

    t1 = PythonOperator(
        task_id="flaky_extract",
        python_callable=flaky_extract
    )
    t2 = PythonOperator(
        task_id="validate_data",
        python_callable=validate_data
    )
    t3 = PythonOperator(
        task_id="transform_and_load",
        python_callable=transform_and_load
    )

    t1 >> t2 >> t3
'''

with open("/content/airflow/dags/orders_pipeline_v2.py", "w") as f:
    f.write(dag_code_v2)

print("✅ DAG v2 written with file-based counter fix")

✅ DAG v2 written with file-based counter fix


In [14]:
!airflow dags test orders_pipeline_v2 2026-07-07

[2026-07-07T16:29:12.577+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.schemas
[2026-07-07T16:29:12.578+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.tables
[2026-07-07T16:29:12.579+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.types
[2026-07-07T16:29:12.579+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.constraints
[2026-07-07T16:29:12.579+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.defaults
[2026-07-07T16:29:12.579+0000] {plugins.py:37} INFO - setup plugin alembic.autogenerate.comments
[2026-07-07T16:29:12.742+0000] {dagbag.py:545} INFO - Filling up the DagBag from /content/airflow/dags
[2026-07-07T16:29:13.032+0000] {example_kubernetes_executor.py:39} WARNING - The example_kubernetes_executor example DAG requires the kubernetes provider. Please install it with: pip install apache-airflow[cncf.kubernetes]
[2026-07-07T16:29:13.039+0000] {example_python_decorator.py:80} WARNING - The virtalenv_python 